# Modèle Prédictif — Meta Score T+1 mois

**Objectif :** prédire le `meta_score` d'un archetype le mois prochain.

**Méthode :**
- Features temporelles : lags T-1/T-2/T-3, momentum (delta), accélération, rolling mean
- Features de rang : percentile dans le mois (invariant au niveau global)
- Walk-forward cross-validation : fenêtre glissante de 9 mois (évite le leakage + distribution shift)
- Ensemble : 70% stabilité (naïf) + 30% modèle Ridge → meilleure ρ de Spearman
- Métrique principale : **Spearman ρ** (corrélation de rang) — plus robuste que R² ici

**Découverte clé :** la méta est "sticky" — Spearman ρ naïf = +0.508 en 2026.
Le modèle apporte un signal marginal sur les archetypes en transition.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

con = sqlite3.connect('../data/yugioh.db')
ms = pd.read_sql("""
    SELECT month, archetype, share, avg_placement, meta_score
    FROM meta_scores ORDER BY month, archetype
""", con)
ms['month'] = pd.to_datetime(ms['month'])
print(f'meta_scores : {len(ms):,} lignes, {ms["month"].nunique()} mois, {ms["archetype"].nunique()} archetypes')

meta_scores : 462 lignes, 30 mois, 122 archetypes


## 1. Construction des features

In [2]:
## 1. Features statiques + banlist temporelle (TOK-22/23)

# ── Banlist courante (snapshot) → trend_ratio global ─────────────────────────
cards_bl = pd.read_sql("""
    SELECT archetype,
           SUM(CASE WHEN ban_tcg = 'Forbidden'    THEN 1 ELSE 0 END) AS n_banned,
           SUM(CASE WHEN ban_tcg = 'Limited'       THEN 1 ELSE 0 END) AS n_limited,
           SUM(CASE WHEN ban_tcg = 'Semi-Limited'  THEN 1 ELSE 0 END) AS n_semi,
           COUNT(*) AS n_cards
    FROM cards
    WHERE archetype IS NOT NULL
    GROUP BY archetype
""", con)
cards_bl['trend_ratio'] = (cards_bl['n_limited'] + cards_bl['n_semi']) / (cards_bl['n_cards'] + 1)

static = cards_bl[['archetype', 'trend_ratio', 'n_cards']].copy()

# ── Banlist temporelle — statut par (archetype, mois) ─────────────────────────
banl_feat = pd.read_sql("""
    SELECT month, archetype, n_forbidden, n_limited AS bl_n_limited,
           n_semi, ban_severity, was_hit_recent, ban_severity_t1
    FROM banlist_features
""", con)
banl_feat['month'] = pd.to_datetime(banl_feat['month'])

# TOK-23 — trend_ratio mensuel : ratio de cartes sous restriction à chaque mois
# Remplace le trend_ratio statique (global) par un signal temporel
banl_feat = banl_feat.merge(static[['archetype', 'n_cards']], on='archetype', how='left')
banl_feat['trend_ratio_monthly'] = (
    (banl_feat['bl_n_limited'] + banl_feat['n_semi']) / (banl_feat['n_cards'].fillna(1) + 1)
)

print(f'static           : {len(static)} archetypes')
print(f'banlist_features : {len(banl_feat)} lignes (mois × archetype)')
print(f'trend_ratio_monthly — max={banl_feat["trend_ratio_monthly"].max():.3f}  mean={banl_feat["trend_ratio_monthly"].mean():.4f}')

static           : 624 archetypes
banlist_features : 3660 lignes (mois × archetype)
trend_ratio_monthly — max=0.333  mean=0.0098


## 2. Features temporelles — lags, momentum, rang

In [3]:
g = ms.sort_values(['archetype', 'month']).groupby('archetype')

# Lags T-1, T-2, T-3
for col in ['meta_score', 'share', 'avg_placement']:
    for lag in [1, 2, 3]:
        ms[f'{col}_t{lag}'] = g[col].shift(lag)

# Momentum : delta 1m, 2m + accélération
ms['delta_1m'] = ms['meta_score'] - ms['meta_score_t1']
ms['delta_2m'] = ms['meta_score'] - ms['meta_score_t2']
ms['accel']    = ms['delta_1m'] - (ms['meta_score_t1'] - ms['meta_score_t2'])

# Rolling mean 3m (sur T-1, T-2, T-3 — pas de leakage)
ms['roll_mean_3m'] = g['meta_score'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=2).mean()
)

# Rang percentile dans le mois (invariant au niveau global du meta_score)
ms['rank_month'] = ms.groupby('month')['meta_score'].rank(pct=True)
ms['rank_t1']    = ms.groupby('month')['meta_score_t1'].rank(pct=True)

# Target : meta_score T+1 et delta T+1
ms['meta_score_next'] = g['meta_score'].shift(-1)
ms['month_next']      = g['month'].shift(-1)
ms['delta_next']      = ms['meta_score_next'] - ms['meta_score']

# Filtrer : T+1 doit être le mois calendaire suivant
ms_clean = ms.dropna(subset=['meta_score_next', 'meta_score_t1', 'delta_next'])
ms_clean = ms_clean[(ms_clean['month_next'] - ms_clean['month']).dt.days.between(25, 35)]

# Merge features statiques (trend_ratio) + banlist temporelle par (archetype, month)
dataset = ms_clean.merge(static[['archetype', 'trend_ratio']], on='archetype', how='left')
dataset = dataset.merge(
    banl_feat[['archetype', 'month', 'ban_severity', 'n_forbidden',
               'was_hit_recent', 'ban_severity_t1', 'trend_ratio_monthly']],
    on=['archetype', 'month'], how='left'
).fillna(0)

# TOK-24 — months_since_debut : ancienneté de l'archetype dans meta_scores
debut = ms.groupby('archetype')['month'].min().rename('debut_month').reset_index()
dataset = dataset.merge(debut, on='archetype', how='left')
dataset['months_since_debut'] = (
    (dataset['month'].dt.year  - dataset['debut_month'].dt.year)  * 12
    + (dataset['month'].dt.month - dataset['debut_month'].dt.month)
)

FEATURE_COLS = [
    # Niveaux absolus
    'meta_score', 'meta_score_t1', 'meta_score_t2', 'meta_score_t3',
    # Momentum
    'delta_1m', 'delta_2m', 'accel', 'roll_mean_3m',
    # Part et placement
    'share', 'share_t1', 'avg_placement', 'avg_placement_t1',
    # Rang relatif dans le mois (stationnaire)
    'rank_month', 'rank_t1',
    # Banlist temporelle (TOK-22) — statut à la date de la prédiction
    'ban_severity', 'n_forbidden', 'was_hit_recent', 'ban_severity_t1',
    # TOK-23 — trend_ratio mensuel (remplace le statique)
    'trend_ratio_monthly',
    # TOK-24 — ancienneté de l'archetype (en mois depuis first apparition)
    'months_since_debut',
]

print(f'Dataset : {len(dataset)} exemples, {len(FEATURE_COLS)} features')
print(f'Mois couverts : {dataset["month"].min().date()} → {dataset["month"].max().date()}')
print(f'\nDistrib delta_next : mean={dataset["delta_next"].mean():+.4f}  std={dataset["delta_next"].std():.4f}')
print(f'\nmonths_since_debut : mean={dataset["months_since_debut"].mean():.1f}  max={dataset["months_since_debut"].max():.0f}')
print(f'trend_ratio_monthly non-nul : {(dataset["trend_ratio_monthly"] > 0).sum()} exemples')

Dataset : 202 exemples, 20 features
Mois couverts : 2024-02-01 → 2026-05-01

Distrib delta_next : mean=-0.0077  std=0.1108

months_since_debut : mean=6.8  max=28
trend_ratio_monthly non-nul : 47 exemples


## 3. Walk-Forward Cross-Validation (fenêtre 9 mois)

In [4]:
WINDOW = 9   # fenêtre glissante en mois
W_MODEL = 0.3  # poids modèle dans l'ensemble (70% naïf + 30% modèle = meilleur en WF-CV)

months = sorted(dataset['month'].unique())
wf_results = []

for i in range(WINDOW, len(months)):
    train_months = months[max(0, i - WINDOW):i]
    test_month   = months[i]

    train_mask = dataset['month'].isin(train_months)
    test_mask  = dataset['month'] == test_month

    X_tr = dataset.loc[train_mask, FEATURE_COLS].values
    X_te = dataset.loc[test_mask,  FEATURE_COLS].values
    y_tr = dataset.loc[train_mask, 'delta_next'].values
    y_te = dataset.loc[test_mask,  'meta_score_next'].values
    cur  = dataset.loc[test_mask,  'meta_score'].values

    if len(X_te) < 3:
        continue

    scaler = StandardScaler()
    model  = Ridge(alpha=50.0)
    model.fit(scaler.fit_transform(X_tr), y_tr)
    pred_model   = cur + model.predict(scaler.transform(X_te))
    pred_blended = W_MODEL * pred_model + (1 - W_MODEL) * cur

    rho_model,   _ = spearmanr(y_te, pred_model)
    rho_blended, _ = spearmanr(y_te, pred_blended)
    rho_naive,   _ = spearmanr(y_te, cur)
    r2_blended     = r2_score(y_te, pred_blended)

    wf_results.append({
        'month': test_month, 'n_test': len(y_te),
        'rho_model': rho_model, 'rho_blended': rho_blended, 'rho_naive': rho_naive,
        'r2_blended': r2_blended,
        'beats_naive': rho_blended > rho_naive,
    })

wf_df = pd.DataFrame(wf_results)

print(f'Walk-Forward CV — {len(wf_df)} mois testés (fenêtre={WINDOW}m)\n')
print(f'{"":25s}  {"ρ mean":>8}  {"ρ median":>9}')
print(f'  Naïf "no change"         {wf_df["rho_naive"].mean():+.3f}     {wf_df["rho_naive"].median():+.3f}')
print(f'  Modèle Ridge (δ)         {wf_df["rho_model"].mean():+.3f}     {wf_df["rho_model"].median():+.3f}')
print(f'  Ensemble 30/70           {wf_df["rho_blended"].mean():+.3f}     {wf_df["rho_blended"].median():+.3f}')

won = wf_df['beats_naive'].sum()
print(f'\nEnsemble bat naïf : {won}/{len(wf_df)} mois ({100*won/len(wf_df):.0f}%)')

recent = wf_df[wf_df['month'] >= '2026-01-01']
if len(recent):
    print(f'\nMois 2026+ uniquement ({len(recent)}) :')
    print(f'  Naïf       ρ = {recent["rho_naive"].mean():+.3f}')
    print(f'  Ensemble   ρ = {recent["rho_blended"].mean():+.3f}')

print('\nDétail par mois :')
for _, r in wf_df.iterrows():
    marker = "✓" if r['beats_naive'] else "✗"
    print(f'  {r["month"].strftime("%Y-%m")} n={r["n_test"]:2.0f}  '
          f'ρ_blend={r["rho_blended"]:+.3f}  ρ_naive={r["rho_naive"]:+.3f}  {marker}')

Walk-Forward CV — 15 mois testés (fenêtre=9m)

                             ρ mean   ρ median
  Naïf "no change"         +0.253     +0.464
  Modèle Ridge (δ)         +0.186     +0.348
  Ensemble 30/70           +0.306     +0.275

Ensemble bat naïf : 7/15 mois (47%)

Mois 2026+ uniquement (5) :
  Naïf       ρ = +0.508
  Ensemble   ρ = +0.441

Détail par mois :
  2024-11 n= 5  ρ_blend=-0.300  ρ_naive=-0.500  ✓
  2025-02 n= 5  ρ_blend=+0.051  ρ_naive=-0.921  ✓
  2025-03 n= 5  ρ_blend=+0.100  ρ_naive=+0.300  ✗
  2025-04 n= 7  ρ_blend=-0.036  ρ_naive=-0.036  ✗
  2025-05 n= 9  ρ_blend=+0.025  ρ_naive=+0.025  ✗
  2025-08 n= 4  ρ_blend=+1.000  ρ_naive=+0.949  ✓
  2025-09 n= 7  ρ_blend=+0.500  ρ_naive=+0.500  ✗
  2025-10 n= 5  ρ_blend=+0.900  ρ_naive=+0.800  ✓
  2025-11 n= 7  ρ_blend=-0.324  ρ_naive=-0.324  ✗
  2025-12 n= 6  ρ_blend=+0.464  ρ_naive=+0.464  ✗
  2026-01 n=10  ρ_blend=+0.248  ρ_naive=+0.220  ✓
  2026-02 n=18  ρ_blend=+0.275  ρ_naive=+0.652  ✗
  2026-03 n=36  ρ_blend=+0.702  ρ_naiv

## 4. Feature importance

In [5]:
# Entraîner sur tout le dataset pour la feature importance
scaler_full = StandardScaler()
rf_full = RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=2, random_state=42)
rf_full.fit(
    scaler_full.fit_transform(dataset[FEATURE_COLS].values),
    dataset['delta_next'].values
)
importances = pd.Series(rf_full.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

print('Feature importance — RF entraîné sur tout le dataset (target = delta_next) :')
for feat, imp in importances.items():
    bar = '█' * int(imp * 50)
    print(f'  {feat:22s}  {imp:.3f}  {bar}')

Feature importance — RF entraîné sur tout le dataset (target = delta_next) :
  meta_score              0.358  █████████████████
  delta_1m                0.176  ████████
  avg_placement           0.121  ██████
  accel                   0.099  ████
  meta_score_t1           0.048  ██
  share                   0.027  █
  months_since_debut      0.026  █
  rank_t1                 0.024  █
  share_t1                0.023  █
  rank_month              0.022  █
  delta_2m                0.020  
  roll_mean_3m            0.017  
  avg_placement_t1        0.016  
  meta_score_t3           0.011  
  meta_score_t2           0.009  
  ban_severity_t1         0.001  
  n_forbidden             0.001  
  ban_severity            0.000  
  trend_ratio_monthly     0.000  
  was_hit_recent          0.000  


## 5. Prédictions mois prochain (archetypes actifs récemment)

In [6]:
import datetime

# Entraîner sur la fenêtre glissante la plus récente
train_months = months[-WINDOW:]
train_mask   = dataset['month'].isin(train_months)
X_tr_final   = dataset.loc[train_mask, FEATURE_COLS].values
y_tr_final   = dataset.loc[train_mask, 'delta_next'].values

scaler_pred = StandardScaler()
model_pred  = Ridge(alpha=50.0)
model_pred.fit(scaler_pred.fit_transform(X_tr_final), y_tr_final)

# État le plus récent de chaque archetype — filtrer à < 4 mois (actifs récemment)
RECENCY_MONTHS = 4
cutoff = dataset['month'].max() - pd.DateOffset(months=RECENCY_MONTHS)
last_ms = (ms_clean[ms_clean['month'] >= cutoff]
           .sort_values('month')
           .groupby('archetype')
           .last()
           .reset_index())

# Merge trend_ratio_monthly (banlist du dernier mois connu)
last_banl = (banl_feat.sort_values('month')
             .groupby('archetype')
             .last()
             .reset_index()[['archetype', 'ban_severity', 'n_forbidden',
                              'was_hit_recent', 'ban_severity_t1', 'trend_ratio_monthly']])
last_ms = last_ms.merge(last_banl, on='archetype', how='left').fillna(0)

# TOK-24 — months_since_debut au moment de la prédiction
last_ms = last_ms.merge(debut, on='archetype', how='left')
last_ms['months_since_debut'] = (
    (last_ms['month'].dt.year  - last_ms['debut_month'].dt.year)  * 12
    + (last_ms['month'].dt.month - last_ms['debut_month'].dt.month)
)

for col in FEATURE_COLS:
    if col not in last_ms.columns:
        last_ms[col] = 0.0

X_pred_final = last_ms[FEATURE_COLS].fillna(0).values
deltas       = model_pred.predict(scaler_pred.transform(X_pred_final))

last_ms['pred_delta']      = deltas
last_ms['pred_meta_score'] = (last_ms['meta_score'] + W_MODEL * deltas).clip(lower=0)
last_ms['pred_direction']  = np.where(deltas > 0.01, '↑', np.where(deltas < -0.01, '↓', '→'))
last_ms['data_month']      = last_ms['month'].dt.strftime('%Y-%m')

preds = last_ms[['archetype','data_month','meta_score','pred_delta','pred_meta_score','pred_direction']]
preds = preds.sort_values('pred_meta_score', ascending=False)

print(f'=== PRÉDICTIONS — TOP 20 ARCHETYPES (entraîné sur {train_months[0].strftime("%Y-%m")}→{train_months[-1].strftime("%Y-%m")}) ===\n')
print(preds.head(20).to_string(index=False, float_format='{:.4f}'.format))

print('\n=== ARCHETYPES EN HAUSSE (delta > 0.01) ===')
print(preds[preds['pred_direction']=='↑'][['archetype','meta_score','pred_delta','pred_meta_score']].head(10).to_string(index=False, float_format='{:.4f}'.format))

# Sauvegarder
con2 = sqlite3.connect('../data/yugioh.db')
con2.execute("DROP TABLE IF EXISTS meta_predictions")
con2.execute("""CREATE TABLE meta_predictions (
    archetype TEXT, data_month TEXT, meta_score_current REAL,
    pred_delta REAL, pred_meta_score REAL, pred_direction TEXT, computed_at TEXT
)""")
out = preds.copy()
out.columns = ['archetype','data_month','meta_score_current','pred_delta','pred_meta_score','pred_direction']
out['computed_at'] = datetime.date.today().isoformat()
out.to_sql('meta_predictions', con2, if_exists='append', index=False)
con2.commit(); con2.close()
print(f'\nSauvegardé : {len(out)} prédictions dans meta_predictions')
print(f'Note : Spearman ρ validé en WF-CV = +{wf_df["rho_blended"].mean():.3f} (moyen)')

=== PRÉDICTIONS — TOP 20 ARCHETYPES (entraîné sur 2025-09→2026-05) ===

            archetype data_month  meta_score  pred_delta  pred_meta_score pred_direction
              Branded    2026-05      0.0838      0.0475           0.0981              ↑
Radiant Typhoon Yummy    2026-03      0.0966     -0.0185           0.0910              ↓
                DoomZ    2026-05      0.0918     -0.0131           0.0879              ↓
          Sky Striker    2026-05      0.0794     -0.0144           0.0751              ↓
            Kewl Tune    2026-05      0.0734      0.0050           0.0749              →
            Mitsurugi    2026-05      0.0770     -0.0102           0.0739              ↓
            Dracotail    2026-05      0.0617      0.0388           0.0733              ↑
                Odion    2026-04      0.0750     -0.0230           0.0681              ↓
              Elfnote    2026-05      0.0656      0.0045           0.0670              →
      Radiant Typhoon    2026-05      

## 6. Modèle séquentiel AR(1) par archetype (TOK-25)

Alternative à la régression cross-sectionnelle Ridge.
Pour chaque archetype, un modèle AR(1) est entraîné sur sa propre série temporelle → prédiction 1 mois ahead.

- AR(1) : `meta_score_t = c + φ × meta_score_{t-1}` — bien adapté aux séries "sticky"
- Fallback naïf si moins de 4 observations
- Blend final : 40% AR(1) + 30% Ridge + 30% naïf

In [7]:
from statsmodels.tsa.ar_model import AutoReg
import warnings

MIN_OBS = 4  # minimum d'observations pour fitter AR(1)

def forecast_ar1(series):
    """AR(1) 1-step forecast. Fallback naïf si données insuffisantes."""
    s = series.dropna()
    if len(s) < MIN_OBS:
        return float(s.iloc[-1]) if len(s) > 0 else 0.0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            res = AutoReg(s.values, lags=1, old_names=False).fit()
            return float(res.predict(start=len(s), end=len(s))[0])
        except Exception:
            return float(s.iloc[-1])


# Pivot meta_score → séries temporelles par archetype
pivot = ms.pivot_table(index='month', columns='archetype', values='meta_score')

WINDOW_AR = 9
W_AR      = 0.4   # poids AR(1) dans le blend final
W_RIDGE   = 0.3   # poids Ridge
W_NAIVE   = 0.3   # poids naïf "no change"

all_months = sorted(pivot.index.unique())
ar_results = []

for i in range(WINDOW_AR, len(all_months)):
    train_months_ar = all_months[max(0, i - WINDOW_AR):i]
    test_month_ar   = all_months[i]

    # On a besoin du meta_score actuel (T) pour comparer avec T+1
    test_rows = dataset[dataset['month'] == test_month_ar]
    if len(test_rows) < 3:
        continue

    ar_preds = {}
    for arch in test_rows['archetype']:
        series = pivot.loc[pivot.index.isin(train_months_ar), arch]
        ar_preds[arch] = forecast_ar1(series)

    test_rows = test_rows.copy()
    test_rows['pred_ar1'] = test_rows['archetype'].map(ar_preds).fillna(test_rows['meta_score'])

    y_true = test_rows['meta_score_next'].values
    cur    = test_rows['meta_score'].values
    p_ar1  = test_rows['pred_ar1'].values

    # Ridge prediction pour ce mois (depuis wf_results)
    ridge_row = next((r for r in wf_results if r['month'] == test_month_ar), None)

    rho_ar1,   _ = spearmanr(y_true, p_ar1)
    rho_naive, _ = spearmanr(y_true, cur)

    ar_results.append({
        'month':      test_month_ar,
        'n_test':     len(y_true),
        'rho_ar1':    rho_ar1,
        'rho_naive':  rho_naive,
        'rho_ridge':  ridge_row['rho_model']   if ridge_row else np.nan,
        'rho_blend_old': ridge_row['rho_blended'] if ridge_row else np.nan,
    })

ar_df = pd.DataFrame(ar_results)

# Blend 3 modèles : AR(1) + Ridge + Naïf
print(f'Walk-Forward CV (AR) — {len(ar_df)} mois\n')
print(f'  Naïf       ρ = {ar_df["rho_naive"].mean():+.3f}')
print(f'  Ridge only ρ = {ar_df["rho_ridge"].mean():+.3f}')
print(f'  AR(1) only ρ = {ar_df["rho_ar1"].mean():+.3f}')
print(f'  Blend prev ρ = {ar_df["rho_blend_old"].mean():+.3f}  (70% naïf + 30% Ridge)')

recent_ar = ar_df[ar_df['month'] >= '2026-01-01']
if len(recent_ar):
    print(f'\nMois 2026+ ({len(recent_ar)}) :')
    print(f'  Naïf       ρ = {recent_ar["rho_naive"].mean():+.3f}')
    print(f'  AR(1) only ρ = {recent_ar["rho_ar1"].mean():+.3f}')
    print(f'  Ridge only ρ = {recent_ar["rho_ridge"].mean():+.3f}')

print('\nDétail par mois (AR1 vs naïf) :')
for _, r in ar_df.iterrows():
    marker = "✓" if r['rho_ar1'] > r['rho_naive'] else "✗"
    print(f'  {r["month"].strftime("%Y-%m")}  AR1={r["rho_ar1"]:+.3f}  naïf={r["rho_naive"]:+.3f}  {marker}')

Walk-Forward CV (AR) — 16 mois

  Naïf       ρ = +0.212
  Ridge only ρ = +0.186
  AR(1) only ρ = +0.141
  Blend prev ρ = +0.306  (70% naïf + 30% Ridge)

Mois 2026+ (5) :
  Naïf       ρ = +0.508
  AR(1) only ρ = +0.243
  Ridge only ρ = +0.302

Détail par mois (AR1 vs naïf) :
  2024-10  AR1=-1.000  naïf=-0.400  ✗
  2024-11  AR1=+0.300  naïf=-0.500  ✓
  2025-02  AR1=+0.821  naïf=-0.921  ✓
  2025-03  AR1=+0.500  naïf=+0.300  ✓
  2025-04  AR1=+0.739  naïf=-0.036  ✓
  2025-05  AR1=+0.176  naïf=+0.025  ✓
  2025-08  AR1=-0.600  naïf=+0.949  ✗
  2025-09  AR1=-0.179  naïf=+0.500  ✗
  2025-10  AR1=+0.400  naïf=+0.800  ✗
  2025-11  AR1=-0.324  naïf=-0.324  ✗
  2025-12  AR1=+0.203  naïf=+0.464  ✗
  2026-01  AR1=+0.200  naïf=+0.220  ✗
  2026-02  AR1=+0.182  naïf=+0.652  ✗
  2026-03  AR1=+0.361  naïf=+0.697  ✗
  2026-04  AR1=+0.243  naïf=+0.485  ✗
  2026-05  AR1=+0.230  naïf=+0.486  ✗


In [8]:
# Blend search : AR(1) + Ridge + Naïf — grid search sur les poids
# Évaluation sur toutes les prédictions poolées (pas per-month pour éviter biais taille)

from itertools import product as iproduct

# Recollect per-fold predictions
fold_preds_all = []
for i in range(WINDOW, len(all_months)):
    train_months_fold = all_months[max(0, i - WINDOW):i]
    test_month_fold   = all_months[i]
    test_rows_fold = dataset[dataset['month'] == test_month_fold].copy()
    if len(test_rows_fold) < 3:
        continue
    train_mask_fold = dataset['month'].isin(train_months_fold)
    X_tr_fold = dataset.loc[train_mask_fold, FEATURE_COLS].values
    y_tr_fold = dataset.loc[train_mask_fold, 'delta_next'].values
    X_te_fold = test_rows_fold[FEATURE_COLS].values
    sc = StandardScaler(); md = Ridge(alpha=50.0)
    md.fit(sc.fit_transform(X_tr_fold), y_tr_fold)
    test_rows_fold['pred_ridge'] = test_rows_fold['meta_score'].values + md.predict(sc.transform(X_te_fold))
    ar_map = {arch: forecast_ar1(pivot.loc[pivot.index.isin(train_months_fold), arch])
              for arch in test_rows_fold['archetype']}
    test_rows_fold['pred_ar1']   = test_rows_fold['archetype'].map(ar_map).fillna(test_rows_fold['meta_score'])
    test_rows_fold['pred_naive'] = test_rows_fold['meta_score']
    fold_preds_all.append(test_rows_fold[['month', 'archetype', 'meta_score_next',
                                          'pred_naive', 'pred_ridge', 'pred_ar1']])

all_preds_df = pd.concat(fold_preds_all)

grid = []
for w_ar in np.arange(0.0, 0.51, 0.1):
    for w_ridge in np.arange(0.0, 0.51, 0.1):
        w_naive = round(1 - w_ar - w_ridge, 2)
        if w_naive < 0:
            continue
        blend = (w_ar    * all_preds_df['pred_ar1']
                + w_ridge * all_preds_df['pred_ridge']
                + w_naive * all_preds_df['pred_naive'])
        rho, _ = spearmanr(all_preds_df['meta_score_next'], blend)
        grid.append((w_ar, w_ridge, w_naive, rho))

grid.sort(key=lambda x: -x[3])

rho_naive_pool,  _ = spearmanr(all_preds_df['meta_score_next'], all_preds_df['pred_naive'])
rho_ar1_pool,    _ = spearmanr(all_preds_df['meta_score_next'], all_preds_df['pred_ar1'])
rho_old_blend, _   = spearmanr(all_preds_df['meta_score_next'],
                                 0.3 * all_preds_df['pred_ridge'] + 0.7 * all_preds_df['pred_naive'])
best_w_ar, best_w_ridge, best_w_naive, best_rho_pool = grid[0]

print('=== Blend grid search (ρ poolé sur toutes les prédictions) ===\n')
print(f'  Naïf seul          ρ = {rho_naive_pool:+.4f}')
print(f'  AR(1) seul         ρ = {rho_ar1_pool:+.4f}')
print(f'  70% Naïf + 30% Ridge  ρ = {rho_old_blend:+.4f}  ← modèle actuel')
print(f'  Meilleur blend     ρ = {best_rho_pool:+.4f}  (AR1={best_w_ar:.1f} Ridge={best_w_ridge:.1f} Naïf={best_w_naive:.1f})')
print(f'\nGain AR(1) : {best_rho_pool - rho_old_blend:+.4f} ρ vs modèle actuel\n')
print('Top 5 blends :')
print(f'  {"w_AR1":>5} {"w_Ridge":>7} {"w_Naïf":>7}  {"ρ":>7}')
for w_a, w_r, w_n, rho in grid[:5]:
    print(f'    {w_a:.1f}     {w_r:.1f}      {w_n:.1f}    {rho:+.4f}')

print(f'''
=== Conclusion TOK-25 ===
AR(1) seul : sous-performe le naïf en méta stable (2026 Kewl Tune dominance).
Blend optimal 3-voies : {best_w_ar:.0%} AR(1) + {best_w_ridge:.0%} Ridge + {best_w_naive:.0%} Naïf → ρ = {best_rho_pool:+.4f}
Gain marginal ({best_rho_pool - rho_old_blend:+.4f} ρ) — conserver le modèle actuel Ridge+Naïf.
AR(1) utile uniquement lors de transitions rapides de méta (avant-2026).
''')

=== Blend grid search (ρ poolé sur toutes les prédictions) ===

  Naïf seul          ρ = +0.6315
  AR(1) seul         ρ = +0.4701
  70% Naïf + 30% Ridge  ρ = +0.6413  ← modèle actuel
  Meilleur blend     ρ = +0.6504  (AR1=0.2 Ridge=0.1 Naïf=0.7)

Gain AR(1) : +0.0091 ρ vs modèle actuel

Top 5 blends :
  w_AR1 w_Ridge  w_Naïf        ρ
    0.2     0.1      0.7    +0.6504
    0.1     0.1      0.8    +0.6500
    0.2     0.0      0.8    +0.6496
    0.1     0.0      0.9    +0.6483
    0.1     0.2      0.7    +0.6469

=== Conclusion TOK-25 ===
AR(1) seul : sous-performe le naïf en méta stable (2026 Kewl Tune dominance).
Blend optimal 3-voies : 20% AR(1) + 10% Ridge + 70% Naïf → ρ = +0.6504
Gain marginal (+0.0091 ρ) — conserver le modèle actuel Ridge+Naïf.
AR(1) utile uniquement lors de transitions rapides de méta (avant-2026).

